# Mobile Game Player Spending Segmentation
## 01 — Synthetic dataset generation

The approved Kaggle source has 3,024 rows and 13 columns. This notebook preserves the source rows, creates simulated rows to reach 250,000 records, adds 10 gameplay/acquisition features, and appends 2,500 exact duplicate rows for the cleaning exercise.

`InstallDate` is generated for **every record**. It is not randomly left blank. One deliberately future date is added as a quality issue; the cleaning notebook should flag only that date. The date field is written in several documented formats so the cleaning parser can be tested.

The source already contains some unavailable `LastPurchaseDate` values. The generator preserves those source gaps and does not add more, because inventing or hiding purchase dates would weaken date-based analysis. Other columns still contain deliberate missingness for cleaning practice.

Gameplay variables are simulated with simple relationships to player segment and activity, so EDA can reveal patterns. These relationships are **teaching assumptions**, not measured behavior or causal evidence about real players.

**Output:** `mobile_game_players_expanded_dirty.csv` (250,000 rows; 23 columns).


## 1. Download Kaggle data

In [ ]:
# KaggleHub downloads the public dataset. If automatic download fails, upload the CSV or ZIP.
!pip -q install kagglehub
from pathlib import Path
import kagglehub
import zipfile
DATASET_HANDLE = 'pratyushpuri/mobile-game-in-app-purchases-dataset-2025'
try:
    downloaded_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
    matches = list(downloaded_dir.rglob('mobile_game_inapp_purchases.csv'))
    if not matches:
        matches = list(downloaded_dir.rglob('*.csv'))
    if not matches:
        raise FileNotFoundError('No CSV file was found in the Kaggle download.')
    SOURCE_CSV = matches[0]
except Exception as error:
    print('Automatic download failed:', repr(error))
    print('Upload the Kaggle ZIP or CSV file.')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No source file was uploaded.')
    uploaded_path = Path(next(iter(uploaded)))
    if uploaded_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(uploaded_path) as archive:
            archive.extractall('kaggle_source')
        matches = list(Path('kaggle_source').rglob('mobile_game_inapp_purchases.csv'))
        if not matches:
            matches = list(Path('kaggle_source').rglob('*.csv'))
        if not matches:
            raise FileNotFoundError('No CSV file was found inside the uploaded ZIP.')
        SOURCE_CSV = matches[0]
    else:
        SOURCE_CSV = uploaded_path
print('Using source file:', SOURCE_CSV)


## 2. Imports, settings, and source check

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20261109
FINAL_ROWS = 250_000
DUPLICATE_RATE = 0.01
OUTPUT_FILENAME = 'mobile_game_players_expanded_dirty.csv'
source = pd.read_csv(SOURCE_CSV)
required_columns = [
    'Age', 'Gender', 'Country', 'Device', 'GameGenre', 'SessionCount',
    'AverageSessionLength', 'SpendingSegment', 'InAppPurchaseAmount',
    'FirstPurchaseDaysAfterInstall', 'PaymentMethod', 'LastPurchaseDate'
]
missing_columns = [column for column in required_columns if column not in source.columns]
if missing_columns:
    raise ValueError(f'The selected source CSV is missing required columns: {missing_columns}')
if source.empty:
    raise ValueError('The selected Kaggle source CSV is empty.')
number_of_duplicates = round(FINAL_ROWS * DUPLICATE_RATE)
if len(source) + number_of_duplicates >= FINAL_ROWS:
    raise ValueError('The source has too many rows for the requested 250,000-row output.')
print('Original Kaggle file:', source.shape)
print('Original columns:', source.columns.tolist())
display(source.head())


### Before generation: inspect the Kaggle source

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
source['SpendingSegment'].value_counts().reindex(['Minnow', 'Dolphin', 'Whale']).plot.bar(ax=axes[0])
axes[0].set_title('Original source: spending segments')
axes[0].set_xlabel('Segment')
axes[0].set_ylabel('Players')

sessions = pd.to_numeric(source['SessionCount'], errors='coerce').dropna()
axes[1].hist(sessions, bins=20, edgecolor='white')
axes[1].set_title('Original source: session counts')
axes[1].set_xlabel('Sessions')
axes[1].set_ylabel('Records')
plt.tight_layout()
plt.show()
print('Missing values in the original file:')
display(source.isna().sum().to_frame('missing_rows'))


### Step 1 — Decide how many rows to create

We keep the Kaggle rows, make simulated rows, and reserve 1% of the final dataset for deliberate duplicates.

In [ ]:
# Same seed = same random choices each time you run the notebook.
rng = np.random.default_rng(SEED)
number_of_simulated = FINAL_ROWS - len(source) - number_of_duplicates
print('Original rows:', len(source))
print('Simulated rows to create:', number_of_simulated)
print('Duplicate rows to add later:', number_of_duplicates)


### Step 2 — Make simulated player rows

We sample existing players **with replacement**. That means a source player may be selected more than once. The small changes below make numeric values vary, but these rows remain simulated—not new measured players.

In [ ]:
# Copy source patterns into a new table.
simulated = source.iloc[
    rng.integers(0, len(source), size=number_of_simulated)
].reset_index(drop=True).copy()

# Slightly vary age while keeping normal values in a plausible range.
age = pd.to_numeric(simulated['Age'], errors='coerce')
simulated['Age'] = np.where(
    age.notna(), np.clip(np.rint(age + rng.normal(0, 3, len(simulated))), 13, 75), np.nan
)

# Vary gameplay sessions and session length a little.
sessions = pd.to_numeric(simulated['SessionCount'], errors='coerce').fillna(8).to_numpy()
simulated['SessionCount'] = np.maximum(
    1, np.rint(sessions * rng.lognormal(0, 0.25, len(simulated)))
).astype(int)

duration = pd.to_numeric(simulated['AverageSessionLength'], errors='coerce').fillna(15).to_numpy()
simulated['AverageSessionLength'] = (duration * rng.lognormal(0, 0.18, len(simulated))).round(2)

print('Simulated table shape:', simulated.shape)
display(simulated.head())


### Step 3 — Combine rows and create the 10 extra features

First combine the original and simulated rows. Then create the extra features for **every row**, so the original rows will not be blank in these newly added columns. The formulas are simple simulation assumptions linked to activity or game genre.

In [ ]:
data = pd.concat([source, simulated], ignore_index=True)
# Use one consistent ID format for every player; IDs do not reveal row origin.
player_numbers = rng.integers(0, 10**15, size=len(data), dtype=np.int64)
data['UserID'] = ['PLAYER' + str(number).zfill(15) for number in player_numbers]
n = len(data)
sessions = pd.to_numeric(data['SessionCount'], errors='coerce').fillna(8).clip(lower=1).to_numpy()
genre = data['GameGenre'].astype('string').fillna('')
segment_text = data['SpendingSegment'].astype('string').str.strip().str.lower()
tier = segment_text.map({'minnow': 0, 'dolphin': 1, 'whale': 2}).fillna(0).to_numpy()

# Parse source dates explicitly before creating installation dates.
def parse_source_dates(series):
    text = series.astype('string').str.strip().str.replace('T', ' ', regex=False)
    parsed = pd.Series(pd.NaT, index=series.index, dtype='datetime64[ns]')
    for date_format in [
        '%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%m/%d/%Y %H:%M:%S',
        '%m/%d/%Y', '%d-%m-%Y %H:%M:%S', '%d-%m-%Y'
    ]:
        parsed = parsed.fillna(pd.to_datetime(text, format=date_format, errors='coerce'))
    return parsed

# 1. Marketing source for the player's install.
data['AcquisitionChannel'] = rng.choice(
    ['Organic', 'Paid Social', 'Search Ads', 'Influencer', 'Referral', 'Cross-promotion'],
    size=n, p=[.25, .24, .14, .12, .13, .12]
)
# 2. Install date: always create a valid date, using LastPurchaseDate when available.
REFERENCE_DATE = pd.Timestamp('2025-08-13')
last_purchase = parse_source_dates(data['LastPurchaseDate']).clip(upper=REFERENCE_DATE)
days_before_purchase = rng.integers(1, 901, size=n)
install_from_purchase = last_purchase - pd.to_timedelta(days_before_purchase, unit='D')
fallback_date = REFERENCE_DATE - pd.to_timedelta(rng.integers(1, 901, size=n), unit='D')
data['InstallDate'] = install_from_purchase.where(last_purchase.notna(), fallback_date)
assert data['InstallDate'].notna().all(), 'Generator error: InstallDate must be populated for every row.'

# 3-6. Activity and engagement. More engaged tiers have modestly higher activity.
active_lambda = np.clip(3.0 + 0.95 * np.log1p(sessions) + 0.8 * tier, 1, 25)
data['ActiveDays30d'] = np.clip(rng.poisson(active_lambda), 1, 30).astype(int)
data['PlayerLevel30d'] = np.maximum(
    1, np.rint(1 + data['ActiveDays30d'].to_numpy() * rng.uniform(.45, 1.45, n))
).astype(int)
# Ad viewing is somewhat higher for lower-spending tiers; this is a teaching assumption.
ad_shift = np.select([tier == 0, tier == 1, tier == 2], [4.0, 1.0, -2.0], default=0.0)
ad_lambda = np.clip(0.8 + 0.35 * data['ActiveDays30d'].to_numpy() + ad_shift, .3, 18)
data['AdsWatched30d'] = rng.poisson(ad_lambda).astype(int)
puzzle_bonus = genre.str.contains('Puzzle', case=False, regex=False).fillna(False).to_numpy() * 1.5
moba_bonus = genre.str.contains('MOBA', case=False, regex=False).fillna(False).to_numpy() * 1.5
data['SocialActions30d'] = rng.poisson(
    np.clip(.8 + .35 * data['ActiveDays30d'].to_numpy() + puzzle_bonus + moba_bonus + .35 * tier, .2, 25)
).astype(int)
# 7-8. Participation is related to activity and, weakly, to spending tier.
guild_probability = np.clip(.10 + .025 * data['ActiveDays30d'].to_numpy() + .06 * tier, .10, .80)
push_probability = np.clip(.28 + .022 * data['ActiveDays30d'].to_numpy() + .045 * tier, .20, .85)
data['GuildMember'] = np.where(rng.random(n) < guild_probability, 'Yes', 'No')
data['PushNotificationsOptIn'] = np.where(rng.random(n) < push_probability, 'Yes', 'No')
# 9. Purchase frequency reflects spending tier and activity; simulated, not observed.
purchase_base = np.select([tier == 0, tier == 1, tier == 2], [.10, .80, 1.90], default=.10)
purchase_lambda = np.clip(purchase_base + .05 * data['ActiveDays30d'].to_numpy(), .05, 8)
data['PurchaseCount30d'] = rng.poisson(purchase_lambda).astype(int)
# 10. Win rate remains bounded between 0 and 1.
data['WinRate'] = np.clip(rng.beta(4, 6, n) + rng.normal(0, .04, n), 0, 1).round(3)
print('Combined table shape:', data.shape)
print('Generated InstallDate coverage:', f"{data['InstallDate'].notna().mean():.1%}")
print('New feature columns:', ['AcquisitionChannel','InstallDate','ActiveDays30d','PlayerLevel30d','AdsWatched30d','SocialActions30d','GuildMember','PushNotificationsOptIn','PurchaseCount30d','WinRate'])


### Step 4 — Add data-quality problems on purpose
These are practice examples for cleaning: missing values in several other fields, inconsistent labels, mixed date formats, mixed units/types, duplicates, outliers, and impossible values. `InstallDate` is deliberately **not** made missing; the mixed formats must remain parseable.


In [ ]:
# Helper: randomly choose rows for one deliberate issue.
def choose_rows(rate):
    count = max(1, int(rate * len(data)))
    return rng.choice(data.index.to_numpy(), size=count, replace=False)

# A. Missing values in several columns, at different rates.
missing_rates = {
    'Age': .025, 'Gender': .018, 'Country': .035,
    'AverageSessionLength': .04, 'AcquisitionChannel': .05,
    'FirstPurchaseDaysAfterInstall': .12, 'PaymentMethod': .045,
'InAppPurchaseAmount': .025,
    'SessionCount': .02, 'ActiveDays30d': .015,
}
for column, rate in missing_rates.items():
    data.loc[choose_rows(rate), column] = np.nan

# B. Different spellings / capitalization for some categories.
label_variations = {
    'Device': {'Android': 'android', 'iOS': 'IOS'},
    'Country': {'UAE': 'United Arab Emirates', 'USA': 'usa', 'UK': 'United Kingdom'},
    'GameGenre': {'Battle Royale': 'BattleRoyale', 'Action RPG': 'action rpg'},
    'SpendingSegment': {'Whale': 'whale', 'Dolphin': 'DOLPHIN', 'Minnow': 'Minnow '},
    'PaymentMethod': {'PayPal': 'Paypal', 'Credit Card': 'CreditCard', 'Google Pay': 'GooglePay'},
}
for column, replacements in label_variations.items():
    selected = choose_rows(.02)
    data.loc[selected, column] = data.loc[selected, column].map(
        lambda value: replacements.get(value, value) if pd.notna(value) else value
    )

# C. Mixed date formats. Every non-null date remains a recognizable date.
def write_mixed_date_formats(column):
    parsed = parse_source_dates(data[column])
    formatted = parsed.dt.strftime('%Y-%m-%d %H:%M:%S').astype(object)
    valid_index = parsed.dropna().index.to_numpy()
    n_slash = min(int(.10 * len(data)), len(valid_index))
    slash_index = rng.choice(valid_index, size=n_slash, replace=False)
    remaining = np.setdiff1d(valid_index, slash_index)
    n_day_first = min(int(.08 * len(data)), len(remaining))
    day_first_index = rng.choice(remaining, size=n_day_first, replace=False)
    formatted.loc[slash_index] = parsed.loc[slash_index].dt.strftime('%m/%d/%Y')
    formatted.loc[day_first_index] = parsed.loc[day_first_index].dt.strftime('%d-%m-%Y')
    data[column] = formatted

write_mixed_date_formats('InstallDate')
write_mixed_date_formats('LastPurchaseDate')
assert data['InstallDate'].notna().all(), 'Date-format step must not create blank InstallDate values.'
# D. Mixed units / types: some durations become seconds-as-text, some amounts include '$'.
data['AverageSessionLength'] = data['AverageSessionLength'].astype(object)
selected = choose_rows(.01)
seconds = pd.to_numeric(data.loc[selected, 'AverageSessionLength'], errors='coerce') * 60
data.loc[selected, 'AverageSessionLength'] = seconds.round().astype('Int64').astype(str) + ' sec'

data['InAppPurchaseAmount'] = data['InAppPurchaseAmount'].astype(object)
selected = choose_rows(.01)
amount = pd.to_numeric(data.loc[selected, 'InAppPurchaseAmount'], errors='coerce').fillna(0)
data.loc[selected, 'InAppPurchaseAmount'] = '$' + amount.round(2).astype(str)

# E. Impossible values and outliers.
selected = choose_rows(.003)
data.loc[selected, 'Age'] = -rng.integers(1, 10, len(selected))
selected = choose_rows(.002)
data.loc[selected, 'SessionCount'] = -rng.integers(1, 30, len(selected))
selected = choose_rows(.001)
data.loc[selected, 'AverageSessionLength'] = rng.uniform(240, 900, len(selected))
selected = choose_rows(.001)
data.loc[selected, 'WinRate'] = rng.uniform(1.2, 2.0, len(selected))
data.loc[data.index[0], 'InstallDate'] = '2035-01-01'  # Deliberately in the future.

print('Quality issues injected. Current shape:', data.shape)
print('InstallDate blanks before cleaning:', int(data['InstallDate'].isna().sum()))
print('A single future InstallDate is intentional and will be flagged during cleaning.')


### Step 5 — Add duplicates and save

We copy 1% of rows exactly. The duplicates are intentional, so you can detect and handle them during cleaning.

In [ ]:
duplicates = data.sample(n=number_of_duplicates, random_state=SEED + 77)
data = pd.concat([data, duplicates], ignore_index=True)
assert len(data) == FINAL_ROWS, f'Expected {FINAL_ROWS:,} rows, got {len(data):,}'
assert data.shape[1] >= 20, 'The dataset needs at least 20 columns.'
assert data['InstallDate'].notna().all(), 'InstallDate must be populated in the dirty output.'
assert int(data.duplicated().sum()) == number_of_duplicates, 'Unexpected duplicate count.'
data.to_csv(OUTPUT_FILENAME, index=False)
print('Final size:', data.shape)
print('Exact duplicate rows:', int(data.duplicated().sum()))
print('Blank InstallDate rows:', int(data['InstallDate'].isna().sum()))
print('Saved as:', OUTPUT_FILENAME)
expanded = data  # Keep the same variable name used in the review cells below.


## 4. After generation: compare before and after

In [ ]:
# Clean label variations for the chart only; the dataset itself stays unchanged.
def clean_segment_for_plot(series):
    return (series.astype('string').str.strip().str.lower()
            .map({'minnow': 'Minnow', 'dolphin': 'Dolphin', 'whale': 'Whale'}))

# Use consistent colors for the source and expanded dataset.
source_color = '#3182bd'
expanded_color = '#fd8d3c'

base_share = (clean_segment_for_plot(source['SpendingSegment'])
              .value_counts(normalize=True)
              .reindex(['Minnow', 'Dolphin', 'Whale']))
expanded_share = (clean_segment_for_plot(expanded['SpendingSegment'])
                  .value_counts(normalize=True)
                  .reindex(['Minnow', 'Dolphin', 'Whale']))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Compare proportions of each spending segment.
segment_comparison = pd.DataFrame({
    'Original Kaggle': base_share,
    'Expanded': expanded_share
})
segment_comparison.plot.bar(
    ax=axes[0, 0], color=[source_color, expanded_color]
)
axes[0, 0].set_title('Spending-segment share: original vs expanded')
axes[0, 0].set_xlabel('Segment')
axes[0, 0].set_ylabel('Proportion')

# 2. Compare session-count distributions.
for label, values, color in [
    ('Original Kaggle', source['SessionCount'], source_color),
    ('Expanded', expanded['SessionCount'], expanded_color),
]:
    values = pd.to_numeric(values, errors='coerce').dropna()
    axes[0, 1].hist(values, bins=30, density=True, alpha=.55, label=label, color=color)
axes[0, 1].set_title('Session counts: original vs expanded')
axes[0, 1].set_xlabel('Sessions')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Convert intentional values such as "600 sec" to minutes for plotting only.
def duration_minutes(series):
    text = series.astype('string').str.strip()
    seconds = text.str.extract(r'^([0-9.]+)\s*sec$', expand=False)
    numbers = pd.to_numeric(text, errors='coerce')
    seconds_mask = seconds.notna()
    numbers.loc[seconds_mask] = (
        pd.to_numeric(seconds[seconds_mask], errors='coerce') / 60
    )
    return numbers

# 3. Compare session duration.
for label, values, color in [
    ('Original Kaggle', source['AverageSessionLength'], source_color),
    ('Expanded', expanded['AverageSessionLength'], expanded_color),
]:
    values = duration_minutes(values).dropna()
    axes[1, 0].hist(values, bins=30, density=True, alpha=.55, label=label, color=color)
axes[1, 0].set_title('Session duration (minutes): original vs expanded')
axes[1, 0].set_xlabel('Minutes')
axes[1, 0].set_ylabel('Density')
axes[1, 0].legend()

# 4. Compare missing-value rates in columns that existed in Kaggle.
shared_columns = [column for column in source.columns if column in expanded.columns]
missing_comparison = pd.DataFrame({
    'Original Kaggle': source[shared_columns].isna().mean(),
    'Expanded': expanded[shared_columns].isna().mean(),
})
missing_comparison.plot.bar(
    ax=axes[1, 1], color=[source_color, expanded_color]
)
axes[1, 1].set_title('Missing-value rates in shared columns')
axes[1, 1].set_ylabel('Fraction')
axes[1, 1].set_xlabel('Column')
axes[1, 1].tick_params(axis='x', rotation=70)

plt.tight_layout()
plt.show()


## 5. Inspect and download

In [ ]:
print('Final dimensions:', expanded.shape)
print('Exact duplicate rows:', int(expanded.duplicated().sum()))
print('Missing values (%):')
display((expanded.isna().mean() * 100).round(2).to_frame('missing_percent'))
print('Target labels (some variations are deliberate):')
display(expanded['SpendingSegment'].value_counts(dropna=False).to_frame('records'))


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILENAME)
except ImportError:
    print('Not running in Colab; output is saved in the current folder:', OUTPUT_FILENAME)


## What is original and what is generated?
- Original source: Kaggle CSV, expected to contain 3,024 records and 13 columns.
- Final dirty dataset: 250,000 records and 23 columns, including 2,500 exact duplicates.
- Original source rows are retained; simulated rows are created by resampling and varying selected values.
- Ten generated features: `AcquisitionChannel`, `InstallDate`, `ActiveDays30d`, `PlayerLevel30d`, `AdsWatched30d`, `SocialActions30d`, `GuildMember`, `PushNotificationsOptIn`, `PurchaseCount30d`, and `WinRate`.
- `InstallDate` is generated for every record; its formats vary but its values are valid except for one deliberate future date.
- Other deliberate quality issues: missing values in several columns, category spelling/case variants, mixed date formats, seconds-as-text session lengths, currency-text purchase amounts, exact duplicate rows, negative age/session counts, unusually high session lengths, win rates above 1, and one future installation date.

All added records and generated relationships are synthetic practice data, not new observations from NovaPlay.
